## [CS149 · PA2] 自己写线程池：三种调度的同步设计

> 2026-10-22 · 来源：CS149 项目复习（真源 modules/408/CS149.md）

- **三种任务系统演进**：每次 run() 都建线程（开销主导，轻负载反而可赢）→ 线程池 + 自旋（延迟低但空占执行资源）→ 线程池 + 条件变量休眠（省资源，需处理唤醒开销）。
- **条件变量纪律**：必须在 `while` 循环里等条件（防虚假唤醒；条件可能在你拿到锁之前已被别的线程满足），判断前先拿锁。
- **「任务完成」的判定用计数，不用队列空**：队列空只说明任务被领走，可能还在执行。用「已完成计数 == 本批总数」，由最后一个完成的线程 notify 主线程。
- **主线程也在抢核**：run() 里自旋等待会占掉一个执行上下文，直接拖慢 worker —— 主线程也该睡。
- **原子 vs 锁**：动态抢任务用原子计数器即可，无需互斥锁；biggraphs 的 BFS 用 compare-and-swap（先读一次、明显会失败就不发 CAS，因为 dist 只会从 -1 变正数不会变回来）。
- **DAG 依赖调度**：等待集合 + 就绪队列双结构，每个 launch 记剩余任务数，归零时给后继的入度减一，入度归零推入就绪队列 —— TBB / Spark / PyTorch 调度器的同构问题。


## Raft 的并发设计：一把大锁 + applyCh 生产者-消费者（互斥与生产者消费者的工程版）

> 2026-10-22 · 来源：6.824 项目复习（`modules/408/6824.md` §3.5，实践仓 `learning-408/raft/src/raft/raft.go`，已过测 2A–2C）

- **我的理解**：待补充
- **知识点**：
  - **一把大锁**：整个 raft 节点的全部共享状态（term / votedFor / logs / nextIndex / matchIndex…）只由**一把互斥锁 `rf.mu`** 保护。理由：raft 各字段耦合极深（一个操作同时碰三四个字段），细粒度锁的锁序设计成本远超收益；且「正确性 > 性能」。这正是**管程（monitor）思想**的工程形态——锁 + 一组不变式，进出临界区由统一入口保证。
  - **绝不在持锁时做慢操作（死锁防线）**：发 RPC、写磁盘都在**锁外**。代码模式是三段式：`Lock → 快照参数 → Unlock → rpc Call（慢）→ Lock → 验证世界没变 → 提交结果 → Unlock`。若锁内发 RPC，对方 handler 也要抢同一把锁，网络一卡即全组死锁。
  - **回锁必须重验（跨阻塞点作废缓存世界观）**：Unlock 到再 Lock 之间状态可能已被改（如已 step down），所以回锁后先查 `rf.role != Leader { return }` 再提交。这与 PV 操作里「P 之后必须重新检查条件」同源。
  - **applyCh = 生产者-消费者的工业版**：协议 goroutine 只负责推高 `committedIndex`（生产），applier goroutine 轮询 `committedIndex > lastApplied` 并把命令推上 `applyCh`（消费）。408 的「缓冲区」从一个环形数组变成了「committedIndex 与 lastApplied 之间的日志区间」——**共享变量 + 互斥锁实现了同一条同步语义**。
  - **先记账后投递**：applier 是「先 `lastAppliedIndex += 1` 再发 applyCh」；反过来（先发再加）在消费方慢时会重复投递——正是生产者-消费者「计数与操作必须原子成对」的要点。

- **易错点**：局部变量 `voteCount` 被 5 个 goroutine 并发 `+=` 却不需要 atomic——因为每次自增都在锁内。**判断要不要加同步原语，看的是访问是否被同一把锁覆盖，而不是「它是不是共享变量」**（这是比背「共享变量要加锁」更深一层的判据）。
- **联动**：goroutine 与 std::thread 的线程模型对照见 `10-进程与线程.ipynb`。

## [CS149] 内存一致性与 lock-free：acquire/release、ABA

> 2026-10-22 · 来源：CS149 项目复习（真源 modules/408/CS149.md）

- **coherence vs consistency**：coherence 管「单个地址」上所有核看到的写顺序；consistency 管「跨地址」的全局读写顺序约定。两回事，别混用。
- **C++ memory_order 五档**：relaxed（只保证原子性）/ acquire + release（一对单向围栏，锁的最小构件）/ acq_rel / seq_cst（全局总序，最贵，默认）。**经典坑：双检查锁定（DCLP）不用 acquire/release 时，可能先看到非空指针、后看到未构造完的对象。**
- **mutex 自带 acquire/release 语义** —— 锁保护的数据不会观察到半成品，不是运气，是围栏。
- **SPSC 队列**：单生产者单消费者，head/tail 各归其主，所有权划分消灭同步，连 CAS 都不用 —— 「消除共享 > 加锁」的教科书案例。
- **ABA 问题**：CAS 只认值不认历史 —— pop 时 head 从 A 走到 B 又回到同地址的 A2，CAS 骗过、拿到悬垂指针。修复：tagged pointer（指针带版本号）；工业级还有 hazard pointer / epoch reclamation。
- **纪律**：lock-free 是最后手段，先量测锁竞争是否真是瓶颈，再上；正确性论证难度是锁的十倍。
